In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(dotenv_path=Path(r"C:\Users\shonr\OneDrive - Tekframeworks\Secret_keys\.env"), override=False)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_LLM_DEPLOYMENT = os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

def validate_config():
    missing=[]
    for k,v in {
        "AZURE_OPENAI_ENDPOINT":AZURE_OPENAI_ENDPOINT,
        "AZURE_OPENAI_API_KEY":AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_LLM_DEPLOYMENT":AZURE_OPENAI_LLM_DEPLOYMENT,
        "AZURE_OPENAI_API_VERSION":AZURE_OPENAI_API_VERSION
    }.items():
        if not v:
            missing.append(k)
    if missing:
        raise ValueError(f"Missing environment variables: {missing}")
    print("CONFIG LOADED SUCCESSFULLY")


## 📚 Notebook Overview

This notebook introduces core **LLM usage patterns** and terminology that you'll need throughout Module 5:

* **Vocabulary first**: We'll define patterns like LLM calls, chat with memory, workflows, agents, and agentic systems.
* **Two perspectives**: Each pattern is shown using both the **OpenAI SDK directly** and with **LangChain** as a helper layer.
* **Preparation for the use case**: The main use case (proposal generation) will be implemented later in *pure SDK*. As homework, you'll port it to LangChain using the patterns learned here.

### Prerequisites

* Basic understanding of LLM API calls (prompt → response)
* Familiarity with RAG (Retrieval-Augmented Generation) from earlier modules
* Python programming fundamentals
* Azure AI Foundry / OpenAI API credentials configured

## 🔧 Setup and Configuration

### Important Configuration Notes

Before running this notebook:

1. **Configure credentials**: Your `.env` file should include:
   - `AZURE_OPENAI_ENDPOINT`: Your Azure OpenAI resource endpoint
   - `AZURE_OPENAI_API_KEY`: Your API key
   - `AZURE_OPENAI_LLM_DEPLOYMENT`: Your model deployment name
   - `AZURE_OPENAI_API_VERSION`: The API version (e.g., 2024-12-01-preview)

2. **Example `.env` format**:
   ```bash
   AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
   AZURE_OPENAI_API_KEY=your-api-key-here
   AZURE_OPENAI_LLM_DEPLOYMENT=gpt-4o-mini
   AZURE_OPENAI_API_VERSION=2024-12-01-preview
   ```

3. **Never commit real API keys** to version control!

In [ ]:
# Install / upgrade the packages used in this notebook
!pip install -U pip
!pip install -U jupyter ipykernel "langchain>=1.0" "langchain-core>=1.0" "langchain-community>=0.3" "langchain-openai>=0.3" pydantic

# Standard libraries
import os
import sys
import json
import requests
import subprocess
from typing import List, Dict, Any

# Modern LangChain imports for v1+
try:
    from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
    from langchain.tools import tool
    from langchain.agents import create_agent
    from langchain_openai import AzureChatOpenAI
    LANGCHAIN_AVAILABLE = True
    print("✅ Modern LangChain components imported successfully")
except ImportError as e:
    LANGCHAIN_AVAILABLE = False
    print(f"⚠️ LangChain not fully available: {e}")

print("📦 Imports complete!")


In [ ]:
# Validate configuration (will raise error if not configured)
# Comment out this line if you want to skip validation during development
validate_config()

---

## 🪜 Concept Ladder: From LLM Calls to Agentic Systems

Before diving into code, let's establish a **shared vocabulary** for LLM usage patterns. These terms will be used throughout Module 5.

### Pattern Definitions

| Pattern | What it is | Typical use |
|---------|-----------|-------------|
| **LLM call** | Single request-response interaction with a language model | One-off tasks: translation, summarization, Q&A |
| **Chat + memory** | Multi-turn conversation where previous messages are included in context | Chatbots, conversational assistants, customer support |
| **Prompt app / template** | Reusable prompt structure with parameters that get filled in | Standardized tasks: email generation, report formatting |
| **Workflow / chain** | Multiple LLM calls in a **fixed sequence** defined by you | Multi-step content generation, ETL-like pipelines |
| **Agent** | LLM that can **choose between tools or steps** at runtime | Dynamic problem-solving, research assistants, code helpers |
| **Agentic system** | Multiple agents or complex control flow with **feedback loops** | Multi-agent collaboration, iterative refinement, approval workflows |
| **RAG** | Retrieve relevant context, then Read and Respond | Question-answering over documents, knowledge bases |

### Progression Diagram

```
Simple → Complex

LLM call → Chat + memory → Prompt app → Workflow → Agent → Agentic system
   ↓            ↓              ↓            ↓          ↓           ↓
Stateless    Context       Parameters   Fixed      Dynamic    Feedback
single       tracking      + reuse      sequence   choices    + loops
```

### Key Distinction: What is an "Agent"?

For this module, an **agent** is:

> **An LLM that can choose between tools, actions, or steps at runtime based on instructions and context.**

This is different from a **workflow** where:
- You (the developer) define every step explicitly
- The sequence is predetermined and doesn't change based on the input

### Where Does RAG Fit In?

You've already learned about **RAG (Retrieval-Augmented Generation)** in earlier modules:

- **RAG pattern**: Retrieve → Read → Respond
- RAG provides **additional context** from external sources (documents, databases, APIs)
- RAG can be used **inside** workflows, agents, or agentic systems
- It's a way to ground LLM responses in factual, up-to-date information

---

## 🔨 Barebones OpenAI SDK – Core Patterns

In this section, we'll implement fundamental LLM patterns using **only the raw OpenAI/Azure SDK** with direct HTTP requests. No frameworks, no abstractions—just the basics.

### 3.1 Single LLM Call (Stateless)

The simplest pattern: send a prompt, get a response. The model has **no memory** of previous interactions.

In [ ]:
def call_llm(prompt: str, temperature: float = 0.0, max_tokens: int = 300) -> str:
    """
    Make a single stateless LLM call using Azure AI Foundry endpoint.
    
    Think of this as: LLM as a function.
    - Input: prompt (string)
    - Output: response (string)
    - No memory: each call is independent
    """
    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_LLM_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    
    headers = {
    "Content-Type": "application/json",
    "api-key": AZURE_OPENAI_API_KEY,
    }
    
    body = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    data = response.json()
    
    return data["choices"][0]["message"]["content"]


# Example: Simple question
response = call_llm("What is the capital of France?")
print("Q: What is the capital of France?")
print(f"A: {response}")
print("\n" + "="*60 + "\n")

# Example: Rewrite task
response = call_llm("Rewrite this in a friendly tone: 'Your request has been received and will be processed.'")
print("Task: Rewrite in friendly tone")
print(f"Result: {response}")

**💡 Try it yourself:**

Modify the code above to ask a different question, such as:
- "Explain quantum computing in one sentence"
- "Translate 'Hello, how are you?' to Spanish"
- "Write a haiku about coding"

Notice that each call is completely independent—there's no memory from previous calls.

### 3.2 Chat + Manual Memory

To enable multi-turn conversations, we need to **manually track and send previous messages** with each request. The model itself has no memory—we're responsible for maintaining context.

In [ ]:
from typing import List, Dict
import requests

def call_llm_with_history(
    messages: List[Dict[str, str]],
    temperature: float = 0.7,
    max_tokens: int = 300
) -> str:
    """
    Make an LLM call with conversation history.
    """

    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_LLM_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"

    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY,
    }

    body = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(url, headers=headers, json=body, timeout=60)

    if not response.ok:
        print("STATUS:", response.status_code)
        print("ERROR:", response.text)
        response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]

In [ ]:
# ============================================================
# TESTING: Conversation with Memory Demo
# ============================================================

print("\n===== MEMORY DEMO: Multi-Turn Conversation =====\n")

# We store conversation manually in a list
# Each message has:
# role → who spoke
# content → what they said
conversation = []

# ------------------------------------------------------------
# TURN 1 — User introduces themselves
# ------------------------------------------------------------
print("STEP 1 → User introduces their name\n")

conversation.append({
    "role": "user",
    "content": "My name is Alice. Nice to meet you!"
})

response1 = call_llm_with_history(conversation)

conversation.append({
    "role": "assistant",
    "content": response1
})

print("User:", conversation[-2]["content"])
print("Assistant:", response1)
print("\nStored Messages:", len(conversation))
print("-" * 50)


# ------------------------------------------------------------
# TURN 2 — Ask model to recall name
# ------------------------------------------------------------
print("\nSTEP 2 → Ask model to remember name\n")

conversation.append({
    "role": "user",
    "content": "What's my name?"
})

response2 = call_llm_with_history(conversation)

conversation.append({
    "role": "assistant",
    "content": response2
})

print("User:", conversation[-2]["content"])
print("Assistant:", response2)
print("\nStored Messages:", len(conversation))
print("-" * 50)


# ------------------------------------------------------------
# TURN 3 — Add new information
# ------------------------------------------------------------
print("\nSTEP 3 → Add preference information\n")

conversation.append({
    "role": "user",
    "content": "I love Python programming."
})

response3 = call_llm_with_history(conversation)

conversation.append({
    "role": "assistant",
    "content": response3
})

print("User:", conversation[-2]["content"])
print("Assistant:", response3)
print("\nStored Messages:", len(conversation))
print("-" * 50)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------
print("\n===== FINAL SUMMARY =====")
print(f"Total conversation messages stored: {len(conversation)}")

print("""
Key Learning:
-------------
The model remembered the user's name and context
ONLY because we kept sending the full conversation history.

LLMs do NOT remember anything on their own.
We simulate memory by passing past messages each time.
""")

**🔑 Key Insight:**

The model doesn't "remember" anything. We're implementing memory by:
1. Storing all previous messages in a list (`conversation`)
2. Sending the **entire conversation history** with each new request
3. Appending the model's response back to the history

This is how every chatbot works under the hood!

### 3.3 Prompt App with Basic Templating

A **prompt app** is a reusable function that:
- Takes parameters (like `topic`, `tone`, `audience`)
- Plugs them into a template
- Calls the LLM
- Returns the result

This makes prompts **reusable and consistent** across your application.

In [ ]:
# ============================================================
# PROMPT APP: Reusable Email Generator
# ============================================================

def generate_email(topic: str, tone: str = "professional", audience: str = "colleagues") -> str:
    """
    Generates a structured email using a reusable prompt template.

    Why this matters:
    -----------------
    Instead of rewriting prompts every time,
    we standardize them into a reusable function.

    This pattern is called a:
        → Prompt Application (Prompt App)

    Parameters:
        topic → what the email is about
        tone → writing style
        audience → who will read it
    """

    # --------------------------------------------------------
    # Prompt Template
    # --------------------------------------------------------
    prompt = f"""
You are an expert communication assistant.

Write a concise email.

Topic: {topic}
Tone: {tone}
Audience: {audience}

Instructions:
- Keep it 3–4 sentences
- Include a subject line
- Use appropriate greeting
- Use appropriate closing
- Keep wording natural and professional

Output format:

Subject: <subject>

<email body>
"""

    # --------------------------------------------------------
    # LLM Call
    # --------------------------------------------------------
    response = call_llm(prompt, temperature=0.7)

    return response.strip()

In [ ]:
# ============================================================
# TESTING THE PROMPT APP
# ============================================================

print("\n========== PROMPT APP DEMO ==========\n")

# ------------------------------------------------------------
# Example 1 — Professional Email
# ------------------------------------------------------------
print("TEST CASE 1 → Professional email to team\n")

email1 = generate_email(
    topic="upcoming team meeting on Q1 planning",
    tone="professional",
    audience="team members"
)

print(email1)
print("\n" + "-" * 60 + "\n")


# ------------------------------------------------------------
# Example 2 — Friendly Email
# ------------------------------------------------------------
print("TEST CASE 2 → Friendly email to stakeholders\n")

email2 = generate_email(
    topic="successful product launch",
    tone="friendly and enthusiastic",
    audience="stakeholders"
)

print(email2)
print("\n" + "-" * 60 + "\n")


# ------------------------------------------------------------
# Teaching Insight
# ------------------------------------------------------------
print("""
Key Concept:
-----------
Same function.
Same template.
Different outputs.

We changed ONLY inputs:
- topic
- tone
- audience

This is the core idea behind Prompt Apps:
→ reusable structured prompts
""")

**💡 Exercise:**

Try adding a new parameter to the `generate_email()` function:
- Add a `length` parameter (e.g., "short", "medium", "long")
- Or add a `language` parameter (e.g., "English", "Spanish")
- Update the prompt template to use this parameter
- Test it with different values

This pattern scales to any kind of parameterized content generation!

---

## 🔗 Mini Workflows (Still Pure SDK)

A **workflow** or **chain** is when you compose multiple LLM calls into a pipeline. Each step feeds into the next, but the sequence is **fixed and predetermined** by your code.

### 4.1 Two-Step Workflow Example

Let's create a simple content generation pipeline:
1. **Step 1**: Generate an outline for a topic
2. **Step 2**: Expand the outline into full content

In [ ]:
def generate_outline(topic: str) -> str:
    """Step 1: Create a simple outline for a topic."""
    prompt = f"""Create a brief outline for a blog post about: {topic}

Requirements:
- 3-4 main sections
- Each section as a bullet point
- Keep it concise

Outline:"""
    
    return call_llm(prompt, temperature=0.3)


def expand_outline(outline: str) -> str:
    """Step 2: Expand an outline into paragraphs."""
    prompt = f"""Take this outline and expand each section into 2-3 sentences:

{outline}

Expanded content:"""
    
    return call_llm(prompt, temperature=0.7, max_tokens=500)


# Run the two-step workflow
print("🔷 STEP 1: Generate Outline")
print("=" * 60)
topic = "Benefits of remote work"
outline = generate_outline(topic)
print(f"Topic: {topic}\n")
print(outline)

print("\n\n🔷 STEP 2: Expand Outline into Content")
print("=" * 60)
expanded_content = expand_outline(outline)
print(expanded_content)

print("\n\n✅ Workflow Complete!")
print("We explicitly controlled the sequence: outline → expand")

### 4.2 Optional Third Step (Formatting / Style)

Workflows can have as many steps as needed. Let's add a third step that refines the style of our content.

In [ ]:
def refine_style(text: str, style: str = "formal") -> str:
    """Step 3: Adjust the writing style of the content."""
    prompt = f"""Rewrite the following text in a {style} style:

{text}

Rewritten version:"""
    
    return call_llm(prompt, temperature=0.5, max_tokens=500)


# Run the full three-step workflow
print("📝 THREE-STEP WORKFLOW DEMO")
print("=" * 60)

# Step 1
topic = "Importance of code reviews"
outline = generate_outline(topic)
print(f"✓ Step 1 complete: Generated outline for '{topic}'")

# Step 2
content = expand_outline(outline)
print(f"✓ Step 2 complete: Expanded outline into content")

# Step 3
refined = refine_style(content, style="friendly and conversational")
print(f"✓ Step 3 complete: Refined style to 'friendly and conversational'")

print("\n" + "=" * 60)
print("FINAL OUTPUT:")
print("=" * 60)
print(refined)

**🔑 Key Point: This is a Workflow, Not an Agent**

Notice that:
- **We** decided the order: outline → expand → refine
- The sequence is **fixed** in our code
- The model doesn't choose which step to do next
- Every input follows the same path

This is deterministic and predictable. Later, we'll see how agents differ by making **dynamic choices** about what to do next.

---

## 🦜 LangChain View: Same Ideas, Just Wrapped

Now let's see how **LangChain** represents these same patterns using higher-level abstractions. LangChain provides utilities that make it easier to build, test, and maintain LLM applications.

### 5.1 From Raw Prompt → PromptTemplate

Remember our `generate_email()` function that used f-strings? LangChain's `PromptTemplate` does the same thing but with additional features like validation and serialization.

**📋 Note:** The following sections require LangChain. Ensure you have installed: `pip install langchain langchain-openai langchain-community`

In [ ]:
# ============================================================
#  PROMPT TEMPLATE DEMO using Langchain
# ============================================================

from langchain_core.prompts import PromptTemplate

print("\nRunning Simplified PromptTemplate Demo\n")

# ------------------------------------------------------------
# 1. PROMPT TEMPLATE
# ------------------------------------------------------------
email_template = PromptTemplate.from_template(
"""Write an email.

Topic: {topic}
Tone: {tone}
Audience: {audience}

Requirements:
- 3–4 sentences
- Include subject line
- Greeting + closing

Email:"""
)

# ------------------------------------------------------------
# 2. FORMAT PROMPT
# ------------------------------------------------------------
prompt = email_template.format(
    topic="Quarterly results review",
    tone="professional",
    audience="executive leadership"
)

print("Generated Prompt")
print("="*60)
print(prompt)
print("="*60)

# ------------------------------------------------------------
# 3. CALL YOUR WORKING FUNCTION (Reliable)
# ------------------------------------------------------------
response = call_llm(prompt)

# ------------------------------------------------------------
# 4. SHOW OUTPUT
# ------------------------------------------------------------
print("\nModel Output")
print("-"*60)
print(response)
print("-"*60)

**💡 Compare:**

- **Our f-string approach** (Section 3.3): Simple, direct, easy to understand
- **LangChain PromptTemplate**: Adds structure, validation, and composability

Both accomplish the same goal. LangChain becomes more valuable as complexity increases.

### 5.2 Tiny Agent Example (One Tool)

Now for the key distinction: an **agent** can make **dynamic choices** about when to use tools.

Unlike workflows where we explicitly call functions in order, an agent decides:
- Should I use a tool?
- Which tool should I use?
- What parameters should I pass?

**🎯 Agent vs Workflow:**

| Workflow | Agent |
|----------|-------|
| Fixed sequence you define | Dynamic decisions based on input |
| Predictable path | Adaptive path |
| Always calls same functions in order | Chooses which tools to use and when |
| Easy to debug and audit | More flexible but harder to predict |

# Langchain Agent with tools demo

In [ ]:
# ============================================================
# LANGCHAIN AGENT — MODERN V1 VERSION
# ============================================================

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import AzureChatOpenAI

# ============================================================
# 1. DEFINE TOOL
# ============================================================

@tool
def calculator(expression: str) -> str:
    """Useful for math calculations. Input must be a valid Python math expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Invalid math expression: {e}"

tools = [calculator]

# ============================================================
# 2. INITIALIZE AZURE CHAT MODEL
# ============================================================

llm = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_LLM_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

# ============================================================
# 3. CREATE AGENT (LangChain v1+)
# ============================================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. "
        "Use the calculator tool whenever arithmetic is required. "
        "For non-math questions, answer directly."
    ),
)

# ============================================================
# 4. HELPER TO EXTRACT FINAL TEXT
# ============================================================

def get_final_text(agent_result):
    """Extract the final assistant text from create_agent() output."""
    try:
        messages = agent_result.get("messages", [])
        if messages:
            last_msg = messages[-1]
            content = getattr(last_msg, "content", last_msg)
            if isinstance(content, list):
                parts = []
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        parts.append(item.get("text", ""))
                    else:
                        parts.append(str(item))
                return "\n".join([p for p in parts if p]).strip()
            return str(content).strip()
    except Exception:
        pass
    return str(agent_result)

# ============================================================
# 5. TEST RUNS
# ============================================================

print("=" * 60)
print("TEST 1 — MATH QUESTION")
print("=" * 60)

result_1 = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 15 multiplied by 7?"}
    ]
})
print(get_final_text(result_1))

print("\n" + "=" * 60)
print("TEST 2 — GENERAL QUESTION")
print("=" * 60)

result_2 = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is machine learning?"}
    ]
})
print(get_final_text(result_2))


---

## LangChain Memory Agent 

This example shows an agent that **remembers user context across turns** and also **uses a tool** when calculation is needed. Keep the same `thread_id` across turns so the agent can retain the conversation context.


In [ ]:
# ============================================================
# LANGCHAIN MEMORY AGENT 
# Remembers user context across turns + uses a tool
# ============================================================

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

# ============================================================
# 1. DEFINE TOOL
# ============================================================

@tool
def price_after_discount(input_text: str) -> str:
    """
    Calculate final price after discount.
    Input format: 'original_price,discount_percent'
    Example: '2000,10'
    """
    try:
        price_str, discount_str = input_text.split(",")
        price = float(price_str.strip())
        discount = float(discount_str.strip())
        final_price = price * (1 - discount / 100)
        return f"Final price after {discount}% discount is {final_price:.2f}"
    except Exception as e:
        return f"Invalid input. Use 'price,discount'. Error: {e}"


tools = [price_after_discount]

# ============================================================
# 2. INITIALIZE AZURE CHAT MODEL
# ============================================================

llm = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_LLM_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0,
)

# ============================================================
# 3. CREATE AGENT WITH MEMORY
# ============================================================

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    system_prompt=(
        "You are a helpful shopping assistant. "
        "Remember user details shared in the conversation. "
        "If the user asks for discount or final price calculation, "
        "use the price_after_discount tool."
    ),
)

# ============================================================
# 4. HELPER TO EXTRACT FINAL TEXT
# ============================================================

def get_final_text(agent_result):
    """Extract the final assistant text from create_agent() output."""
    try:
        messages = agent_result.get("messages", [])
        if messages:
            last_msg = messages[-1]
            content = getattr(last_msg, "content", last_msg)
            if isinstance(content, list):
                parts = []
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        parts.append(item.get("text", ""))
                    else:
                        parts.append(str(item))
                return "".join([p for p in parts if p]).strip()
            return str(content).strip()
    except Exception:
        pass
    return str(agent_result)

# ============================================================
# 5. TEST RUNS — SAME thread_id FOR MEMORY
# ============================================================

thread_config = {"configurable": {"thread_id": "shopping-demo-1"}}

print("=" * 60)
print("TURN 1")
print("=" * 60)
result_1 = agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Ravi. I usually want 15% discount."}]},
    config=thread_config,
)
print(get_final_text(result_1))

print("" + "=" * 60)
print("TURN 2")
print("=" * 60)
result_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What discount did I say I prefer?"}]},
    config=thread_config,
)
print(get_final_text(result_2))

print("" + "=" * 60)
print("TURN 3")
print("=" * 60)
result_3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Calculate final price for 3200 using my preference."}]},
    config=thread_config,
)
print(get_final_text(result_3))

# Run example:
# "My name is Ravi. I usually want 15% discount."
# "What discount did I say I prefer?"
# "Calculate final price for 3200 using my preference."



---

## LangChain Conversational Agent 

This example shows a **multi-turn conversational agent** that keeps continuity across follow-up questions. It uses short-term memory through the same `thread_id`, but no external tools.


In [ ]:

# ============================================================
# LANGCHAIN CONVERSATIONAL AGENT 
# Multi-turn chat assistant
# ============================================================

from langchain.agents import create_agent
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

# ============================================================
# 1. INITIALIZE AZURE CHAT MODEL
# ============================================================

llm = AzureChatOpenAI(
    azure_deployment=AZURE_OPENAI_LLM_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.3,
)

# ============================================================
# 2. CREATE CONVERSATIONAL AGENT
# ============================================================

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=memory,
    system_prompt=(
        "You are a friendly classroom AI assistant. "
        "Answer clearly and conversationally. "
        "Keep continuity across turns. "
        "When the user asks a follow-up, use prior conversation context."
    ),
)

# ============================================================
# 3. HELPER TO EXTRACT FINAL TEXT
# ============================================================

def get_final_text(agent_result):
    """Extract the final assistant text from create_agent() output."""
    try:
        messages = agent_result.get("messages", [])
        if messages:
            last_msg = messages[-1]
            content = getattr(last_msg, "content", last_msg)
            if isinstance(content, list):
                parts = []
                for item in content:
                    if isinstance(item, dict) and item.get("type") == "text":
                        parts.append(item.get("text", ""))
                    else:
                        parts.append(str(item))
                return "".join([p for p in parts if p]).strip()
            return str(content).strip()
    except Exception:
        pass
    return str(agent_result)

# ============================================================
# 4. TEST RUNS — SAME thread_id FOR CONVERSATION CONTINUITY
# ============================================================

thread_config = {"configurable": {"thread_id": "classroom-chat-1"}}

print("=" * 60)
print("TURN 1")
print("=" * 60)
result_1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain AI agents in simple terms."}]},
    config=thread_config,
)
print(get_final_text(result_1))

print("" + "=" * 60)
print("TURN 2")
print("=" * 60)
result_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "How is that different from automation?"}]},
    config=thread_config,
)
print(get_final_text(result_2))

# Run example:
# "Explain AI agents in simple terms."
# "How is that different from automation?"


---

## 🎯 How This Connects to the Use Case

Now that you understand the core patterns, let's talk about **what comes next** in this module.

### The Proposal Generation Use Case

You'll work with a real-world scenario: **building a proposal generation assistant**. This use case will be implemented in **two different ways**:

#### **Approach 1: Workflow (Deterministic)**
- Fixed sequence of steps: outline → draft sections → stitch together → format
- Every proposal follows the same path
- ~7 LLM calls, fully predictable

#### **Approach 2: Agentic (Adaptive)**
- Planner → Drafter → Evaluator loop
- The evaluator provides feedback and drafts are refined iteratively
- 5-9 LLM calls depending on quality feedback


---

## ✅ Checklist & Reflection

### Self-Check: Do You Understand?

Go through this checklist to verify your understanding:

- [ ] I understand the difference between a **single LLM call** and **chat with memory**
- [ ] I can explain what a **"prompt app"** is and why templates are useful
- [ ] I can describe a simple **LLM workflow** using plain Python functions
- [ ] I've seen how **LangChain wraps prompts and workflows** with higher-level abstractions
- [ ] I understand what **"agent"** means in this module (LLM that chooses tools dynamically)
- [ ] I can explain the key difference between a **workflow** and an **agent**
- [ ] I know that **RAG** is a way to provide additional context and can be used in workflows or agents
- [ ] I'm ready to tackle the **proposal generation use case**


### Discussion Questions

Think about these questions (discuss with peers or instructor):

1. What are the **trade-offs** between workflows and agents in terms of:
   - Cost (number of LLM calls)?
   - Predictability and debugging?
   - Flexibility and adaptation?

2. In the upcoming proposal generation use case:
   - What benefits might the **workflow approach** provide?
   - What benefits might the **agentic approach** provide?
   - When would you choose one over the other?

3. How would you incorporate **RAG** into either approach?

---

**Happy coding! 🚀**